# Acorn Plan — 6,000-Image Clean Dataset GPU Training

Trains **YOLOv11s-seg** (Segmentation) at **768px** on the massive 6,027-image clean dataset.

## 🚀 Before you run
1. **Upload Dataset**: Copy **`final_clean_dataset (1).zip`** (1.31 GB) to the home screen of your **Google Drive**.
2. **Enable GPU**: Click **Runtime** → **Change runtime type** → select **T4 GPU** (or A100/V100 if you have Colab Pro) → **Save**.
3. Run all cells from top to bottom.

In [ ]:
# 1. Install Ultralytics + confirm GPU is active
!pip -q install ultralytics
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime to T4 GPU!')

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/final_clean_dataset (1).zip'
if os.path.exists(zip_path):
    print("✅ Found 'final_clean_dataset (1).zip' in your Google Drive home screen!")
else:
    print("❌ Error: Could not find 'final_clean_dataset (1).zip' in your Google Drive.")
    print("   Please make sure the file is named exactly 'final_clean_dataset (1).zip' and sits in your main Google Drive folder.")

In [ ]:
# 3. Extract the 1.3 GB dataset to the local Colab fast SSD
import zipfile

DEST = '/content/dataset'
zip_path = '/content/drive/MyDrive/final_clean_dataset (1).zip'

if os.path.exists(zip_path):
    print("Extracting dataset (this takes 1-2 minutes)...")
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DEST)
    print("✅ Successfully extracted to:", DEST)
    
    # Print statistics to verify
    for split in ('train', 'valid'):
        img_dir = os.path.join(DEST, split, 'images')
        if os.path.exists(img_dir):
            n = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
            print(f'  - {split}: {n} images found')

In [ ]:
# 4. Write a Colab-correct data.yaml for the 6 dataset classes
import yaml

data_yaml = f'''path: {DEST}
train: train/images
val: valid/images

nc: 6
names:
  0: acm
  1: door
  2: floor
  3: room
  4: stairs
  5: walls
'''

with open('/content/data.yaml', 'w') as f:
    f.write(data_yaml)
print("✅ Created /content/data.yaml:")
print(data_yaml)

In [ ]:
# 5. Train YOLOv11s-seg (Segmentation) at 768px on the GPU
from ultralytics import YOLO

# Load a pre-trained segmentation model base
model = YOLO('yolo11s-seg.pt')

# Run optimized training
model.train(
    data='/content/data.yaml',
    epochs=150,            # 150 epochs is perfect for 6,000 images to converge
    imgsz=768,             # Recommended size for sketches and wall lines
    batch=32,              # Change to 16 if Colab runs out of memory (OOM)
    device=0,              # Run on GPU
    optimizer='AdamW',     # Best optimizer for segmentation masks
    patience=25,           # Early stopping safeguard
    project='runs',
    name='yolo11s_acorn_seg',
    exist_ok=True,
    plots=True
)

In [ ]:
# 6. Validate + show final metrics
metrics = model.val()
print('\n🏆 FINAL TRAINING METRICS:')
print('  Mask mAP50    :', round(metrics.seg.map50, 4))
print('  Mask mAP50-95 :', round(metrics.seg.map, 4))
for i, name in enumerate(['acm', 'door', 'floor', 'room', 'stairs', 'walls']):
    try:
        print(f'    {name:12s} AP50={metrics.seg.ap50[i]:.3f}')
    except Exception:
        print(f'    {name:12s} (no val instances)')

In [ ]:
# 7. Copy best weights back to Google Drive (Anti-Disconnect Backup)
import shutil

best_weights = '/content/runs/yolo11s_acorn_seg/weights/best.pt'
drive_dest = '/content/drive/MyDrive/best_room.pt'

if os.path.exists(best_weights):
    print("Backing up best weights directly to your Google Drive...")
    shutil.copy(best_weights, drive_dest)
    print("✅ Saved trained weights to your Google Drive home screen as: best_room.pt!")
    print("   You can now download 'best_room.pt' from your Google Drive!")
else:
    print("❌ Error: Weights file not found at:", best_weights)